In [1]:
# 어떤 설비가 가장 생산성이 좋냐?
# 야간 조의 불량률이 더 높은가?
# 이번 달 생산량이 작년보다 증가했냐?
# 어떤 제품이 가장 많이 팔리냐?
# 공정별로 병목이 어디인가?

In [158]:
# 1. gruopby : 데이터를 그룹으로 나누기
# 2. agg : 여러 집계함수를 동시에 이용가능
# 3. pivot_tabe :행/열 구조로 요약
# 4. transorm : 그룹 통계를 원본데이터에 추가
# 5. fillter : 그룹 조건으로 필터링

In [3]:
import pandas as pd

In [25]:
df_production = pd.read_csv('./data/05_production.csv')

In [26]:
df_q = pd.read_csv('./data/07_quality_inspection.csv',na_values= ['\\N'])

In [27]:
df_production.head(5)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48


In [28]:
df_production.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1872 entries, 0 to 1871
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   production_id    1872 non-null   int64  
 1   equipment_id     1872 non-null   object 
 2   product_code     1872 non-null   object 
 3   production_date  1872 non-null   object 
 4   start_time       1872 non-null   object 
 5   end_time         1872 non-null   object 
 6   target_quantity  1872 non-null   int64  
 7   actual_quantity  1872 non-null   int64  
 8   good_quantity    1872 non-null   int64  
 9   defect_quantity  1872 non-null   int64  
 10  cycle_time       1872 non-null   float64
 11  work_order_no    1872 non-null   object 
 12  lot_no           1872 non-null   object 
 13  operator_id      1872 non-null   object 
 14  shift            1872 non-null   object 
 15  created_at       1872 non-null   object 
 16  updated_at       1872 non-null   object 
dtypes: float64(1),

In [29]:
df_production['production_date']=pd.to_datetime(df_production['production_date'])  # 시간데이터 바꾸는 법

In [30]:
df_q.head()

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
0,1,1,INJ-001,BUMPER-A,2024-01-01 09:41:18,FINAL,PASS,NaN,300.8279,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
1,2,1,INJ-001,BUMPER-A,2024-01-01 08:17:24,FINAL,PASS,NaN,299.7696,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59
2,3,1,INJ-001,BUMPER-A,2024-01-01 08:47:26,FINAL,PASS,NaN,301.0795,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
3,4,1,INJ-001,BUMPER-A,2024-01-01 08:33:03,FINAL,PASS,NaN,302.5384,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
4,5,1,INJ-001,BUMPER-A,2024-01-01 09:46:23,FINAL,PASS,NaN,299.6097,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59


In [31]:
df_q['inspection_time']=pd.to_datetime(df_q['inspection_time'])

In [32]:
df_production.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48


In [35]:
df_production['defect_rate']=df_production['defect_quantity']/df_production['actual_quantity']*100

In [38]:
df_production.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.222222
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.173913
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.426357


In [41]:
# 어떤 설비가 가장 많이 사용되는가?  -> 생산부하가 집중된 설비를 파악 가능 -> 병목 지점을 알 수 있다.

In [49]:
df_production.groupby('equipment_id').size().sort_values(ascending=False)  #같은 설비끼리 그룹화하기 ( 오름차순 )

equipment_id
PRESS-002    478
PRESS-001    468
INJ-002      430
INJ-001      262
ASM-001      234
dtype: int64

In [52]:
# cursor,  VS + copilot , antigarabity, VS + clude < 현업에서 많이 쓰는 툴

In [55]:
df_production['equipment_id'] .value_counts()

equipment_id
PRESS-002    478
PRESS-001    468
INJ-002      430
INJ-001      262
ASM-001      234
Name: count, dtype: int64

In [60]:
#어떤 설비가 가장 많은 제품을 생산했는가?
# 고성능 설비를 알 수 있다 -> 생산 계획 시  우선 배정
# 설비 가동률의 기초 데이터로 사용 가능

In [65]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [71]:
df_production.groupby('equipment_id')['actual_quantity'].sum().sort_values(ascending = False)

equipment_id
PRESS-001    52069
INJ-002      51958
PRESS-002    51929
INJ-001      28163
ASM-001      22485
Name: actual_quantity, dtype: int64

In [72]:
# 한번 작업할 때 평균적으로 얼마나 생산하는가?

In [76]:
df_production.groupby('equipment_id')['actual_quantity'].mean().sort_values(ascending = False)

equipment_id
INJ-002      120.832558
PRESS-001    111.258547
PRESS-002    108.638075
INJ-001      107.492366
ASM-001       96.089744
Name: actual_quantity, dtype: float64

In [77]:
# 각 설비별로, 평균 생산량, 표준편차(안정성), 최소, 최대값 

In [84]:
df_summary = df_production.groupby('equipment_id')['actual_quantity'].agg( ['mean','std','min','max','count'] ) # 데이터 집계해서 df 만드는 법

In [89]:
df_summary.rename(columns = {'mean' : '평균','std':'표준편차'})

,평균,표준편차,min,max,count
equipment_id,,,,,
ASM-001,96.089744,16.575114,68,127,234
INJ-001,107.492366,20.022352,65,142,262
INJ-002,120.832558,22.112656,75,157,430
PRESS-001,111.258547,20.685662,67,147,468
PRESS-002,108.638075,20.195916,67,144,478


In [90]:
# 생산성지표, 양품생산능력, 불량발생 정도, 품질 수준 4가지를 한번에 보고 싶다.
# 생산성은 높은데, 불량률도 높은 것? -> 품질개선 프로젝트로 선정할 수 있다.

In [91]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [102]:
df_production.groupby('equipment_id').agg({'actual_quantity' : ['mean','sum'],
                                          'good_quantity':'sum',
                                          'defect_quantity':'sum',
                                          'defect_rate':'mean'})

actual_quantity        good_quantity defect_quantity defect_rate
                        mean    sum           sum             sum        mean
equipment_id                                                                 
ASM-001            96.089744  22485         19759            2726   12.116901
INJ-001           107.492366  28163         25146            3017   10.758940
INJ-002           120.832558  51958         47454            4504    8.701387
PRESS-001         111.258547  52069         46946            5123    9.911201
PRESS-002         108.638075  51929         46423            5506   10.675655

In [104]:
# 설비 + 제품별로 생산량을 확인해보자. -> 설비별로 제품특화 파악 가능
# 교대조 + 설비별로 불량률을 확인해보자.-> 주간조와 야간조의 차이 발견 가능


In [107]:
# 각 설비는 어떤 제품을 주로 생산하는가? -> 설비별로 제품 특화 파악, 교체 작업 빈도 예측, 생산스케쥴링을 최적화 가능

In [108]:
df_production.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308


In [110]:
df_production.groupby(['equipment_id','product_code']).agg(
    생산건수 =('production_id','count'), 총생산량 = ('actual_quantity','sum'),
    평균불량률=('defect_rate','mean')
    
)

생산건수   총생산량      평균불량률
equipment_id product_code                        
ASM-001      BUMPER-A        74   6804  11.096406
             DASH-C          70   6779  12.511699
             DOOR-B          90   8902  12.648909
INJ-001      BUMPER-A       104  10858  11.514531
             DASH-C          85   9536  10.189184
             DOOR-B          73   7769  10.345895
INJ-002      BUMPER-A       147  18053   9.531398
             DASH-C         122  14766   8.109898
             DOOR-B         161  19139   8.391761
PRESS-001    BUMPER-A       171  18757   9.524309
             DASH-C         139  15670  10.210906
             DOOR-B         158  17642  10.066263
PRESS-002    BUMPER-A       152  16663  11.024681
             DASH-C         167  17990  10.481176
             DOOR-B         159  17276  10.546261

In [111]:
# 시계열(time Series) 분석

In [112]:
# 생산량이 증가추세이냐? 감소추세인가?
# 특정요일 또는 특정 월에 패턴이 있냐?
# 갑자기 데이터가 튀는 날이 있냐?
# 월간 목표를 달성했는가?

In [113]:
# 일별분석, 주별분석, 월별 분석

In [114]:
# 월별 생산 추이는 어떠한가?

In [115]:
df_production.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308


In [120]:
df_production['year_month'] = df_production['production_date'].dt.to_period('M') # 년,월,일 데이터를 월데이터로 바꾸기

In [121]:
df_production.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate,year_month
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272,2024-01
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308,2024-01


In [123]:
df_production.groupby('year_month')['actual_quantity'].agg('mean')

year_month
2024-01    111.579872
2024-02    110.282392
2024-03    109.262422
Freq: M, Name: actual_quantity, dtype: float64

In [126]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', 'year_month'],
      dtype='object')

In [124]:
# 주간 조와 야간 조의 성과 차이는?

In [127]:
df_production.groupby('shift').agg({'actual_quantity':'mean', 'cycle_time':'mean', 'defect_rate':'mean' })

,actual_quantity,cycle_time,defect_rate
shift,,,
DAY,109.711538,76.63562,8.815960
NIGHT,111.019231,76.63563,11.629766


In [128]:
#야간 조와 주간 조의 불량률 차이가 있음

In [129]:
# 각 설비는 어떤 제품을 어떤 제품을 얼마나 생산했는가 알고 싶다.

In [130]:
# 설비별로 제품 생산 패턴을 파악해서 설비와 제품의 매칭을 최적화 시키고 싶다

In [136]:
df_result=df_production.groupby(['equipment_id','product_code'])['actual_quantity'].agg('count')

In [137]:
df_result

equipment_id  product_code
ASM-001       BUMPER-A         74
              DASH-C           70
              DOOR-B           90
INJ-001       BUMPER-A        104
              DASH-C           85
              DOOR-B           73
INJ-002       BUMPER-A        147
              DASH-C          122
              DOOR-B          161
PRESS-001     BUMPER-A        171
              DASH-C          139
              DOOR-B          158
PRESS-002     BUMPER-A        152
              DASH-C          167
              DOOR-B          159
Name: actual_quantity, dtype: int64

In [142]:
pd.pivot_table(df_production, index='equipment_id',columns='product_code', values = 'actual_quantity',aggfunc='sum',fill_value = 0)

product_code,BUMPER-A,DASH-C,DOOR-B
equipment_id,,,
ASM-001,6804,6779,8902
INJ-001,10858,9536,7769
INJ-002,18053,14766,19139
PRESS-001,18757,15670,17642
PRESS-002,16663,17990,17276


In [143]:
# 교대조별로 설비품질이 다른가?? -> 특정조의 특성설비 운영능력 평가

In [ ]:
#야간 조의 특정 설비 불랴이 높다 -> 조명이나 교육의 문제
# 모든 조에서 특정 설비의 불량률이 높다라고 나오면? -> 설비 자체 문제?

In [145]:
(pd.pivot_table(df_production, index='shift', columns = 'equipment_id',values = 'defect_rate',aggfunc='mean' )).round(2)

equipment_id,ASM-001,INJ-001,INJ-002,PRESS-001,PRESS-002
shift,,,,,
DAY,10.67,9.36,7.2,8.53,9.34
NIGHT,13.56,12.16,10.2,11.29,12.01


In [148]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', 'year_month'],
      dtype='object')

In [147]:
#여러값을 동시에 집계

In [151]:
(pd.pivot_table(df_production, index='equipment_id', columns='shift', values=['actual_quantity','defect_rate'])).round(2)

actual_quantity         defect_rate       
shift                    DAY   NIGHT         DAY  NIGHT
equipment_id                                           
ASM-001                95.32   96.85       10.67  13.56
INJ-001               105.90  109.08        9.36  12.16
INJ-002               120.17  121.49        7.20  10.20
PRESS-001             110.26  112.26        8.53  11.29
PRESS-002             108.90  108.38        9.34  12.01

In [152]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', 'year_month'],
      dtype='object')

In [156]:
pd.pivot_table(df_production,index='equipment_id',columns='product_code',values='actual_quantity',aggfunc='sum', margins=True) 

product_code,BUMPER-A,DASH-C,DOOR-B,All
equipment_id,,,,
ASM-001,6804,6779,8902,22485
INJ-001,10858,9536,7769,28163
INJ-002,18053,14766,19139,51958
PRESS-001,18757,15670,17642,52069
PRESS-002,16663,17990,17276,51929
All,71135,64741,70728,206604


In [157]:
pd.pivot_table(df_production,index='equipment_id',columns='product_code',values='actual_quantity',aggfunc='sum', margins=True)
# 전체 값도 볼 수 있다

product_code,BUMPER-A,DASH-C,DOOR-B,All
equipment_id,,,,
ASM-001,6804,6779,8902,22485
INJ-001,10858,9536,7769,28163
INJ-002,18053,14766,19139,51958
PRESS-001,18757,15670,17642,52069
PRESS-002,16663,17990,17276,51929
All,71135,64741,70728,206604


In [159]:
#빈도 분석!!

In [160]:
# 빈도 = 우선순위입니다.

In [161]:
# 파레토 법칙 (80:20) -> 20%의 원인이 80%의 문제를 일으킨다

In [162]:
# 설비 사용 빈도 -> 일부 설비에 부하가 집중되었구나.

In [163]:
# 불랼코드 DIM-001 이 전체의 35%다. 이것을 분석했더니 결과가 나왔다.
# -> 조치: 치수 측정 공정에 집중 투자를 하자. -> 이 한가지 문제 해결로 35프로 개선 가능하다

In [164]:
# 어떤 설비가 가장 바쁜가?

In [165]:
df_production['equipment_id'].value_counts()

equipment_id
PRESS-002    478
PRESS-001    468
INJ-002      430
INJ-001      262
ASM-001      234
Name: count, dtype: int64

In [167]:
# 전체 생산에서, 각 설비가 차지하는 비중은? (비율로 질문)
# 부하를 분산할 수 있는 평가가 가능, 어떤 설비를 더 투자해야 할지 우선순위 결정.

In [168]:
# 한 설비가 50% 이상 차지한다. -> 고장 시 생산마비 올 수 있다. 리스크!

In [171]:
df_production['equipment_id'].value_counts(normalize=True)*100

equipment_id
PRESS-002    25.534188
PRESS-001    25.000000
INJ-002      22.970085
INJ-001      13.995726
ASM-001      12.500000
Name: proportion, dtype: float64

In [172]:
# 합격률은 얼마나 되는가?

In [174]:
df_q.head(3)

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
0,1,1,INJ-001,BUMPER-A,2024-01-01 09:41:18,FINAL,PASS,NaN,300.8279,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
1,2,1,INJ-001,BUMPER-A,2024-01-01 08:17:24,FINAL,PASS,NaN,299.7696,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59
2,3,1,INJ-001,BUMPER-A,2024-01-01 08:47:26,FINAL,PASS,NaN,301.0795,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59


In [181]:
df_q['result'].unique()

array(['PASS', 'FAIL'], dtype=object)

In [184]:
df_q['result'].value_counts(normalize=True)

result
FAIL    0.557928
PASS    0.442072
Name: proportion, dtype: float64

In [211]:
df_q.groupby('equipment_id')['defect_code'].value_counts()

equipment_id  defect_code
ASM-001       D001            652
              D002            444
              D005            378
              D003            319
              D006            248
              D009            190
              D010            144
              D004            143
              D007            126
              D008             82
INJ-001       D001            595
              D002            504
              D005            484
              D003            363
              D006            312
              D009            215
              D010            156
              D007            151
              D004            139
              D008             98
INJ-002       D001           1007
              D002            738
              D005            671
              D003            509
              D006            458
              D009            338
              D004            230
              D007            209
              D010    

In [185]:
#어떤 불량이 가장 많은지 확인

In [194]:
#갈 설비에서 제품별로 몇 번 생산했는가?

In [195]:
# 설비 - 제품 매칭 패턴을 파악해서, 교체 작업 회수를 추정한다거나,

In [196]:
# 전용라인 vs 범용라인으로 할건지 전략 수립하는데 활용한다.

In [197]:
# 설비 X 제품 교차표

In [199]:
pd.crosstab( df_production['equipment_id'] , df_production['product_code']  ) #크로스 테이블은 건 수를 구할 때 (pivot이 더 활용도가 높음)

product_code,BUMPER-A,DASH-C,DOOR-B
equipment_id,,,
ASM-001,74,70,90
INJ-001,104,85,73
INJ-002,147,122,161
PRESS-001,171,139,158
PRESS-002,152,167,159


In [203]:
pd.crosstab( df_production['equipment_id'] , df_production['product_code']  ,normalize ='index', margins=True) # 인덱스 기준으로 노말라이즈

product_code,BUMPER-A,DASH-C,DOOR-B
equipment_id,,,
ASM-001,0.316239,0.299145,0.384615
INJ-001,0.396947,0.324427,0.278626
INJ-002,0.341860,0.283721,0.374419
PRESS-001,0.365385,0.297009,0.337607
PRESS-002,0.317992,0.349372,0.332636
All,0.346154,0.311432,0.342415


In [207]:
df_production.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', 'year_month'],
      dtype='object')

In [209]:
df_production.groupby('equipment_id')['actual_quantity'].mean() # 이상태는 데이타 프레임에 안 들어감

equipment_id
ASM-001       96.089744
INJ-001      107.492366
INJ-002      120.832558
PRESS-001    111.258547
PRESS-002    108.638075
Name: actual_quantity, dtype: float64

In [213]:
df_production['설비별 평균 생산량']=df_production.groupby('equipment_id')['actual_quantity'].transform('mean') # 이상태는 데이타 프레임에 안 들어감

In [214]:
df_production

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate,year_month,설비별 평균 생산량
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272,2024-01,107.492366
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308,2024-01,107.492366
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.222222,2024-01,120.832558
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.173913,2024-01,120.832558
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.426357,2024-01,120.832558
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,77.63,WO202403317101,LOT2024033100210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.361111,2024-03,108.638075
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,76.15,WO202403318434,LOT2024033100211,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,16.153846,2024-03,108.638075
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,69.95,WO202403317294,LOT2024033100212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.500000,2024-03,108.638075
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,90.10,WO202403317268,LOT2024033100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,16.528926,2024-03,96.089744


In [215]:
# 평균 대비 편차 -> 실제 생산량 - 평균 생산량

In [216]:
df_production['평균대비 편차']=df_production['actual_quantity']- df_production['설비별 평균 생산량']

In [218]:
# Z-score : (데이터수치 - 평균수치) /표준편차
# 이상치 탐지할 때 사용.
# IQR을 이용한 이상치 탐지, Z - score로 탐지
# Z - score 값이 3보다 크면, 이상치 다.
# Z - score 값이 1.96보다 큰지 -> 신뢰구간 95%
# Z - score 값이 2.58보다 큰지 -> 신뢰구간 99%

In [222]:
df_production['생산수량 표준 편차'] = df_production.groupby('equipment_id')['actual_quantity'].transform('std')

In [224]:
df_production['qty_zscore']=df_production['평균대비 편차']/df_production['생산수량 표준 편차']

In [228]:
df_production.loc[df_production['qty_zscore']>3,]

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,operator_id,shift,created_at,updated_at,defect_rate,year_month,설비별 평균 생산량,평균대비 편차,생산수량 표준 편차,qty_zscore


In [229]:
# 생산 건수가 100건 미만인 설비는 빼고 가져오자

In [232]:
def data_size(x) :
    if len(x)>=100:
        return True
    else :
        return False
        

In [235]:
df_production.groupby('equipment_id').filter(data_size) #filter란 함수는 group함수에 조건을 걸 수 있다

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,operator_id,shift,created_at,updated_at,defect_rate,year_month,설비별 평균 생산량,평균대비 편차,생산수량 표준 편차,qty_zscore
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272,2024-01,107.492366,-26.492366,20.022352,-1.323140
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308,2024-01,107.492366,-29.492366,20.022352,-1.472972
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.222222,2024-01,120.832558,14.167442,22.112656,0.640694
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.173913,2024-01,120.832558,-28.832558,22.112656,-1.303894
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.426357,2024-01,120.832558,8.167442,22.112656,0.369356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.361111,2024-03,108.638075,35.361925,20.195916,1.750944
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,16.153846,2024-03,108.638075,21.361925,20.195916,1.057735
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.500000,2024-03,108.638075,-28.638075,20.195916,-1.418013
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,16.528926,2024-03,96.089744,24.910256,16.575114,1.502871


In [240]:
def get_data(x):
    if x['defect_rate'].mean() >= 5 :
        return True
    else :
        return False

In [241]:
#평균 불량률이 5% 이상인 데이터만 가져오자.

In [242]:
df_production.groupby('equipment_id').filter(get_data)z

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,operator_id,shift,created_at,updated_at,defect_rate,year_month,설비별 평균 생산량,평균대비 편차,생산수량 표준 편차,qty_zscore
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.938272,2024-01,107.492366,-26.492366,20.022352,-1.323140
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.692308,2024-01,107.492366,-29.492366,20.022352,-1.472972
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.222222,2024-01,120.832558,14.167442,22.112656,0.640694
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.173913,2024-01,120.832558,-28.832558,22.112656,-1.303894
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.426357,2024-01,120.832558,8.167442,22.112656,0.369356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.361111,2024-03,108.638075,35.361925,20.195916,1.750944
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,16.153846,2024-03,108.638075,21.361925,20.195916,1.057735
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.500000,2024-03,108.638075,-28.638075,20.195916,-1.418013
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,16.528926,2024-03,96.089744,24.910256,16.575114,1.502871
